# AI & ML for Sustainability — Companion Notebook

This notebook accompanies the lecture slides:  
each section either **demonstrates a concept from the slides** or **shows a pitfall to watch out for**.

You do not need any prior Python experience — every line of code is explained.

---

### What we cover

| Section | What it shows | <br>
|---|---|---|
| 1 | A simple model: Random Forest for water demand |
| 2 | Issue: concept drift — what happens when the world changes |
| 3 | Issue: geographic bias — models that fail in new places |
| 4 | Issue: anomaly detection — spotting the unusual |

---

In [ ]:
# ── Suppress harmless warnings so the output stays clean ──────────────────────
import warnings
warnings.filterwarnings('ignore')

# Core libraries
import numpy as np                        # fast arrays and maths
import pandas as pd                       # spreadsheet-like tables
import matplotlib.pyplot as plt           # plotting

# Machine learning
from sklearn.ensemble import RandomForestRegressor, IsolationForest
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error

# Reproducibility: fixing the random seed means you get the same
# synthetic data every time you run the notebook.
np.random.seed(42)

print('All libraries loaded successfully.')

In [ ]:
# ── Helper functions used throughout the notebook ─────────────────────────────

def section(title):
    """Print a visible section heading in the output."""
    print('\n' + '═' * 70)
    print(f'  {title}')
    print('═' * 70)


def rmse(y_true, y_pred):
    """Root Mean Squared Error — lower is better.
    Returned in the same units as the target variable."""
    return np.sqrt(mean_squared_error(y_true, y_pred))

---
## Section 1 — A simple model: predicting water demand

The slides show that AI is already used in agriculture to reduce water and fertiliser use.

Here we build a **Random Forest** — a popular, robust ML model — to predict daily water demand
from four environmental features: rainfall, temperature, soil moisture, and wind speed.

**Why Random Forest?**
- It is made of many simple decision trees that vote together — more robust than a single tree
- It handles non-linear relationships without any manual tuning
- It tells us which features matter most, giving a first window into *why* it makes predictions
- It is much simpler to use than a deep neural network, yet often performs comparably on tabular data

**Note:** we use synthetic (made-up) data here so you can run everything without any internet connection or real datasets.

In [ ]:
section('1. Random Forest — predicting water demand')

# ── Generate synthetic daily observations ─────────────────────────────────────
# In a real project these would come from sensors or meteorological databases.
n = 500

rainfall      = np.random.uniform(0, 200, n)    # mm/day (0 = dry, 200 = heavy rain)
temperature   = np.random.uniform(10, 40,  n)   # Celsius
soil_moisture = np.random.uniform(0.1, 0.9, n)  # fraction: 0.1 = dry, 0.9 = saturated
wind_speed    = np.random.uniform(0, 35,   n)   # km/h

# ── Define the target: daily water demand (litres per hectare) ────────────────
# Hot + dry + windy days push demand up; heavy rain pushes it down.
# The relationships are intentionally simple and illustrative.
water_demand = (
    20
    - 0.08 * rainfall          # more rain  → less irrigation needed
    + 1.5  * temperature       # hotter     → more evaporation, more demand
    - 30   * soil_moisture     # wet soil   → less irrigation needed
    + 0.4  * wind_speed        # wind dries the soil faster
    + np.random.normal(0, 3, n)  # real-world noise
)

df = pd.DataFrame({
    'rainfall': rainfall,
    'temperature': temperature,
    'soil_moisture': soil_moisture,
    'wind_speed': wind_speed,
    'water_demand': water_demand
})

print(f'Dataset: {n} daily observations')
print(f'Water demand range: {water_demand.min():.1f} – {water_demand.max():.1f} L/ha')
print()
print(df.head(5).round(2))

In [ ]:
# ── Split into training set (80%) and test set (20%) ─────────────────────────
# We train on 400 days, then check predictions on the remaining 100.
# This is fine here because observations are independent daily snapshots —
# there is no time ordering to leak.
features = ['rainfall', 'temperature', 'soil_moisture', 'wind_speed']
X = df[features]
y = df['water_demand']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# ── Train the Random Forest ───────────────────────────────────────────────────
# n_estimators = how many trees to grow (more = more stable, but slower)
# max_depth    = how many splits each tree is allowed to make
rf = RandomForestRegressor(n_estimators=100, max_depth=6, random_state=42)
rf.fit(X_train, y_train)   # training: the model learns from the data

# ── Evaluate on the held-out test set ────────────────────────────────────────
rf_pred = rf.predict(X_test)
rf_rmse = rmse(y_test, rf_pred)

# ── Compare against a naive baseline ─────────────────────────────────────────
# The simplest possible prediction: always guess the training-set mean.
# If our model can't beat this, it's not adding any value.
baseline_pred  = np.full(len(y_test), y_train.mean())
baseline_rmse  = rmse(y_test, baseline_pred)
skill          = 1 - (rf_rmse**2 / baseline_rmse**2)   # 0 = same as baseline, 1 = perfect

print(f'Baseline RMSE (always predict mean): {baseline_rmse:.2f} L/ha')
print(f'Random Forest RMSE:                  {rf_rmse:.2f} L/ha')
print(f'Skill score (vs baseline):           {skill:.2f}  (higher is better; 1.0 = perfect)')

# ── Plot: predicted vs actual ─────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Left: scatter plot of predictions vs truth
axes[0].scatter(y_test, rf_pred, alpha=0.5, s=20, color='#1a6b42')
lims = [min(y_test.min(), rf_pred.min()), max(y_test.max(), rf_pred.max())]
axes[0].plot(lims, lims, 'k--', linewidth=1, label='Perfect prediction')
axes[0].set_xlabel('Actual water demand (L/ha)')
axes[0].set_ylabel('Predicted water demand (L/ha)')
axes[0].set_title('Predicted vs Actual')
axes[0].legend()

# Right: feature importances — which variable matters most?
importances = pd.Series(rf.feature_importances_, index=features).sort_values()
importances.plot(kind='barh', ax=axes[1], color='#1a6b42')
axes[1].set_xlabel('Importance (higher = more influence on predictions)')
axes[1].set_title('Which features drive predictions?')

plt.tight_layout()
plt.show()

print()
print('Feature importance note: temperature and soil moisture dominate because')
print('the target was constructed with the largest coefficients on those variables.')
print('In a real model, importances reveal genuine patterns — or hidden biases.')

### Key takeaway

A Random Forest is a good starting point for most tabular environmental datasets:
it handles non-linear relationships, works with mixed feature types, and gives
feature importances as a first sanity-check on the model.

**Always compare against a simple baseline.** If your fancy model barely beats  
"always predict the average", it is not adding much value.

Feature importances are a useful first look, but they are **not causal** —
a feature can be important without causing the outcome.

---

## Section 2 — Issue: concept drift

**The problem:** environmental data does not stay the same forever.  
A model trained on historical data silently degrades when conditions change —  
a land-use shift, a new heat island, a long drought, or even a sensor replacement
can make past patterns unreliable.

This is called **concept drift**: the statistical relationship between inputs and
outputs changes over time.

In this example we simulate a temperature-like signal that suddenly shifts upward
at time step 130 — mimicking, for instance, a city's urban heat island effect
becoming significant as a new district is built.

A model trained on data from before the shift will be systematically wrong afterward.

In [ ]:
section('2. Concept drift — when the world changes under the model')

# ── Simulate a 300-step temperature-like time series ──────────────────────────
n_steps = 300
t = np.arange(n_steps)

# Base signal: slight upward trend + seasonal wave + noise
signal = 20 + 0.02 * t + 2 * np.sin(t / 8) + np.random.normal(0, 0.8, n_steps)

# ── Inject a sudden level shift at t=130 ─────────────────────────────────────
# This simulates an abrupt change: a new heat island, a sensor swap,
# or a land-use change that permanently raises baseline temperature.
drift_point = 130
signal[drift_point:] += 4.5   # sudden +4.5°C shift

df_drift = pd.DataFrame({'time': t, 'temperature': signal})

# ── Train a simple model on the PRE-DRIFT period only ────────────────────────
# In real life, you would have trained on historical data and not yet seen
# the new regime.  Here we use a linear fit as the stand-in 'deployed model'.
pre_drift   = df_drift[df_drift['time'] < drift_point]
post_drift  = df_drift[df_drift['time'] >= drift_point]

t_train = pre_drift['time'].values.reshape(-1, 1)
y_train_d = pre_drift['temperature'].values
deployed_model = LinearRegression().fit(t_train, y_train_d)

# Predict over the full series with the model trained before the drift
full_pred = deployed_model.predict(t.reshape(-1, 1))

# ── Compute error before and after the drift ──────────────────────────────────
pre_rmse  = rmse(pre_drift['temperature'], full_pred[:drift_point])
post_rmse = rmse(post_drift['temperature'], full_pred[drift_point:])

print(f'Model RMSE before drift (training period): {pre_rmse:.2f}°C')
print(f'Model RMSE after  drift (deployed period): {post_rmse:.2f}°C')
print(f'Error increased {post_rmse/pre_rmse:.1f}× after the level shift.')

# ── Plot ──────────────────────────────────────────────────────────────────────
plt.figure(figsize=(11, 4))
plt.plot(t, signal, linewidth=0.9, label='Actual temperature', color='#1a6b42')
plt.plot(t, full_pred, linewidth=1.2, linestyle='--',
         label='Model prediction (trained before drift)', color='#c8503a')
plt.axvline(drift_point, color='black', linestyle=':', linewidth=1.5,
            label=f'Drift begins (t={drift_point})')
plt.axvspan(drift_point, n_steps, alpha=0.06, color='red')
plt.xlabel('Time step')
plt.ylabel('Temperature (°C)')
plt.title('Concept drift: the deployed model becomes systematically wrong')
plt.legend()
plt.tight_layout()
plt.show()

### Key takeaway

The model was accurate before the drift — then quietly wrong afterward.  
In a real deployment this error would accumulate, potentially for months, before anyone noticed.

**What to do about it:**
- Monitor model error continuously after deployment, not just at launch
- Retrain on a rolling window of recent data (seasonal retraining is common for climate applications)
- If possible, set up automatic alerts when error exceeds a threshold

This is why the slides say: *an AI system is never truly 'finished'.*

---

## Section 3 — Issue: geographic bias

**The problem from the slides:** richer regions have more sensors and more labelled data,
so AI models work better there. Low-income countries get worse predictions — precisely
where good environmental data matters most.

A model trained on European conditions may fail in Africa not because it is badly designed,
but because the statistical patterns it learned simply do not transfer.

Here we simulate two regions with genuinely different climate regimes:
- **Region A** (e.g. temperate Europe): moderate temperatures, frequent rainfall
- **Region B** (e.g. semi-arid Sahel): high temperatures, scarce and erratic rainfall

We train on Region A and evaluate on Region B — a very common real-world scenario.

In [ ]:
section('3. Geographic bias — training on one region, deploying in another')

# ── Region A: temperate European climate ──────────────────────────────────────
n_a = 400
temp_a     = np.random.normal(18, 5, n_a)         # mild: average 18°C
rainfall_a = np.random.normal(70, 20, n_a)        # moderate rainfall
# Target: vegetation index (0–1; higher = more green cover)
# More rain and milder temperatures → more vegetation
veg_a = 0.3 + 0.004 * rainfall_a - 0.005 * temp_a + np.random.normal(0, 0.05, n_a)
veg_a = np.clip(veg_a, 0, 1)

# ── Region B: semi-arid Sahel climate ────────────────────────────────────────
n_b = 200
temp_b     = np.random.normal(34, 4, n_b)         # hot: average 34°C
rainfall_b = np.random.exponential(20, n_b)       # sparse, erratic rainfall
# In this regime, vegetation is far more sensitive to rainfall and much lower overall
veg_b = 0.05 + 0.006 * rainfall_b - 0.001 * temp_b + np.random.normal(0, 0.04, n_b)
veg_b = np.clip(veg_b, 0, 1)

# ── Train exclusively on Region A ────────────────────────────────────────────
X_a = np.column_stack([temp_a, rainfall_a])
X_b = np.column_stack([temp_b, rainfall_b])

model_a = RandomForestRegressor(n_estimators=100, max_depth=5, random_state=42)
model_a.fit(X_a, veg_a)

# ── Evaluate on Region A (familiar) and Region B (new) ───────────────────────
pred_on_a = model_a.predict(X_a)
pred_on_b = model_a.predict(X_b)

rmse_a = rmse(veg_a, pred_on_a)
rmse_b = rmse(veg_b, pred_on_b)

print(f'RMSE on Region A (where the model was trained): {rmse_a:.3f}')
print(f'RMSE on Region B (new climate regime):         {rmse_b:.3f}')
print(f'Error is {rmse_b/rmse_a:.1f}× higher in the new region.')

# ── Plot: predicted vs actual for both regions ────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

for ax, actual, pred, label, color in [
    (axes[0], veg_a, pred_on_a, 'Region A (training region)', '#1a6b42'),
    (axes[1], veg_b, pred_on_b, 'Region B (unseen climate)',  '#c8503a'),
]:
    ax.scatter(actual, pred, alpha=0.4, s=15, color=color)
    lim = [0, 1]
    ax.plot(lim, lim, 'k--', linewidth=1, label='Perfect prediction')
    ax.set_xlim(0, 1); ax.set_ylim(0, 1)
    ax.set_xlabel('Actual vegetation index')
    ax.set_ylabel('Predicted vegetation index')
    ax.set_title(f'{label}\nRMSE = {rmse(actual, pred):.3f}')
    ax.legend(fontsize=9)

plt.suptitle('Same model — very different accuracy depending on where it is applied',
             fontsize=12, y=1.02)
plt.tight_layout()
plt.show()

### Key takeaway

The model performs well where it was trained — and much worse where it wasn't.
This is not a bug in the code; it is a fundamental property of statistical learning.
A model can only generalise as far as its training distribution extends.

**What makes this a fairness issue:**  
Region B is the one that most urgently needs accurate vegetation monitoring for food security.
The communities with the least data end up with the least reliable AI tools.

**What to do about it:**
- Collect labelled data from the target region, even a small amount (a few hundred examples)
- Fine-tune a pre-trained model on local data rather than training from scratch
- Always report where your training data came from and test on held-out regions, not just held-out observations

---

## Section 4 — Issue: anomaly detection

Environmental monitoring produces continuous streams of sensor readings.
Most readings are normal — but the important ones are the exceptions:
a pollution spike, an equipment fault, a methane leak, an extreme flood event.

**Anomaly detection** is the task of automatically flagging readings that
don't fit the expected pattern — without being told in advance what 'abnormal' looks like.

We use **Isolation Forest**, which works by asking: *how many random questions does it take
to isolate this data point from the rest?*  
Normal points blend in and need many questions.  
Anomalies stand out and are isolated quickly with very few questions.

**The hard part (noted in the slides):** genuine anomalies are rare — sometimes 1 in 10,000
readings. Standard accuracy metrics are misleading in this case.
A model that never raises an alarm would be '99.99% accurate' but completely useless.

In [ ]:
section('4. Anomaly detection — spotting the unusual in sensor data')

# ── Simulate a 300-step environmental sensor signal ───────────────────────────
# Normal behaviour: gentle seasonal wave around a baseline of 50 units, with small noise.
# This could represent air quality, water turbidity, or grid voltage.
n_sensor = 300
sensor_t  = np.arange(n_sensor)
signal    = 50 + 3 * np.sin(sensor_t / 12) + np.random.normal(0, 0.8, n_sensor)

# ── Inject known anomalies ────────────────────────────────────────────────────
# In practice these could be: pollution spikes, equipment faults, sensor glitches.
anomaly_idx    = [55, 140, 141, 225, 265]
anomaly_shifts = [12, -14, -11, 16, 10]
for idx, shift in zip(anomaly_idx, anomaly_shifts):
    signal[idx] += shift

sensor_df = pd.DataFrame({'time': sensor_t, 'reading': signal})

# ── Fit Isolation Forest ──────────────────────────────────────────────────────
# contamination = our best guess of what fraction of points are anomalies.
# Setting this too low misses real anomalies; too high creates false alarms.
iso = IsolationForest(contamination=0.03, random_state=42)
labels = iso.fit_predict(sensor_df[['reading']])
# Isolation Forest returns: -1 = anomaly, +1 = normal
sensor_df['anomaly'] = (labels == -1)

# ── Evaluate: did we catch the known anomalies? ───────────────────────────────
# Build a ground-truth flag for comparison
sensor_df['known_anomaly'] = False
sensor_df.loc[anomaly_idx, 'known_anomaly'] = True

detected   = sensor_df[sensor_df['anomaly'] & sensor_df['known_anomaly']]
missed     = sensor_df[~sensor_df['anomaly'] & sensor_df['known_anomaly']]
false_alrm = sensor_df[sensor_df['anomaly'] & ~sensor_df['known_anomaly']]

print(f'Known anomalies:      {len(anomaly_idx)}')
print(f'Detected (correct):   {len(detected)}')
print(f'Missed:               {len(missed)}')
print(f'False alarms:         {len(false_alrm)}')
print()
print('Note: in practice, false alarms and missed events both carry costs.')
print('The contamination parameter is the key trade-off to tune.')

# ── Plot the signal with detected anomalies ───────────────────────────────────
plt.figure(figsize=(12, 4))
plt.plot(sensor_df['time'], sensor_df['reading'],
         linewidth=0.9, color='#1a6b42', label='Sensor reading', zorder=1)

# Mark detected anomalies
detected_points = sensor_df[sensor_df['anomaly']]
plt.scatter(detected_points['time'], detected_points['reading'],
            color='#c8503a', s=60, zorder=3, label='Flagged as anomaly')

# Mark true anomalies (so we can see what was missed)
known_points = sensor_df[sensor_df['known_anomaly']]
plt.scatter(known_points['time'], known_points['reading'],
            edgecolors='black', facecolors='none', s=120, linewidths=1.5,
            zorder=4, label='Known anomaly (ground truth)')

plt.xlabel('Time step')
plt.ylabel('Sensor reading')
plt.title('Isolation Forest anomaly detection — circles = true events, filled = flagged')
plt.legend()
plt.tight_layout()
plt.show()

### Key takeaway

Isolation Forest works without any labelled examples — it just learns what 'normal' looks like
and flags deviations. This makes it practical for environmental monitoring, where labelled
anomaly examples are almost never available.

**The trade-off:** every anomaly detector has two types of mistake:
- **False alarm** — flagging something normal (costly if operators must respond every time)
- **Missed event** — silently ignoring a real problem (potentially much more costly)

The `contamination` parameter controls this trade-off. There is no universally correct value —
it depends on the relative cost of each type of mistake in your specific application.

---

### Key takeaway

In the random split, the test set contains data from the middle of the training period.
The model has already 'seen' neighbouring time points — so predicting nearby values is easy.
This inflates the reported accuracy.

In real deployment you always predict the future from the past — and the time-based split
is the only evaluation that honestly reflects this.

The same principle applies spatially: **test on regions the model has never seen**,
not on random pixels from the same region (which is what Section 3 demonstrated).

---